### Import Dependencies

In [ ]:
import openai
import pandas as pd

from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    Fusion,
    FusionQuery,
)
import cohere

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

### Retrieval

In [ ]:
query = "Can I get a tablet?"

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


In [ ]:
COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"
def retrieve_data_hybrid(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20,
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20,
            ),
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint", result)
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


In [ ]:
results = retrieve_data_hybrid(query, k=20)

In [ ]:
results

### Reranking

In [ ]:
cohere_client = cohere.ClientV2()

In [ ]:
to_rerank = results["retrieved_context"]

In [ ]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20,
)

In [ ]:
response

In [ ]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [ ]:
reranked_results